# Exercice 1 — ÉTUDE DE CAS BLOC 2 (NovaRetail)

## Plan suivi **exactement étape par étape**
1. Sélectionner les observations et variables selon le périmètre défini
2. Réaliser une analyse univariée et bivariée
3. Produire des visualisations pertinentes (3 à 6)
4. Concevoir un tableau de bord synthétique
5. Rédiger une note d’analyse métier (1 à 2 pages)
6. Documenter un carnet technique

> **Périmètre imposé : octobre 2025 uniquement**.

## 0) Préparation : imports, chemins, utilitaires
Notebook autonome (sans dépendances externes obligatoires).

In [ ]:
from __future__ import annotations

import csv
import json
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path
from statistics import mean, median

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def read_csv(path: Path) -> list[dict[str, str]]:
    with path.open("r", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def write_csv(path: Path, rows: list[dict], fieldnames: list[str]) -> None:
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

print("Répertoire courant:", BASE_DIR)

## 1) Sélectionner les observations et variables selon le périmètre défini

### 1.1 Charger les données sources
- `leads_novaretail.csv`
- `campaign_novaretail.json`
- `crm_novaretail.csv`

In [ ]:
leads = read_csv(DATA_DIR / "leads_novaretail.csv")
crm = read_csv(DATA_DIR / "crm_novaretail.csv")
campaigns = json.loads((DATA_DIR / "campaign_novaretail.json").read_text(encoding="utf-8"))

print(f"Leads: {len(leads)} | CRM: {len(crm)} | Campagnes: {len(campaigns)}")

### 1.2 Filtrer les observations hors périmètre
Règle : conserver uniquement les lignes dont `date` est comprise entre `2025-10-01` et `2025-10-31`.

In [ ]:
start = datetime(2025, 10, 1)
end = datetime(2025, 10, 31)

leads_scope = []
for row in leads:
    d = datetime.strptime(row["date"], "%Y-%m-%d")
    if start <= d <= end:
        leads_scope.append(row)

print("Leads avant filtrage:", len(leads))
print("Leads après filtrage (octobre 2025):", len(leads_scope))

### 1.3 Identifier les variables utiles / exclure les non pertinentes

**Variables retenues (avec justification)**
- `lead_id` : clé de jointure entre leads et CRM.
- `date` : vérification du périmètre temporel.
- `channel` : analyse par canal marketing.
- `cost`, `impressions`, `clicks`, `conversions` : calcul des KPI demandés (CTR, conversion, CPL).
- `company_size`, `sector`, `region`, `status` : segmentation métier.

**Variables non prioritaires dans ce périmètre**
- `device` : utile potentiellement, mais non exigé explicitement par les objectifs donnés.
- `campaign_id` : utile pour traçabilité, non indispensable au calcul des KPI demandés.

### 1.4 Exclure doublons / contrôler intégrité / préparer base d’analyse

In [ ]:
# Contrôle doublons sur lead_id côté leads du périmètre
seen = set()
duplicates = []
for row in leads_scope:
    lid = row["lead_id"]
    if lid in seen:
        duplicates.append(lid)
    seen.add(lid)

print("Doublons lead_id détectés:", duplicates if duplicates else "Aucun")

# Jointure leads + CRM
crm_by_lead = {row["lead_id"]: row for row in crm}
merged = []
for lead in leads_scope:
    crm_row = crm_by_lead.get(lead["lead_id"])
    if crm_row:
        merged.append({**lead, **crm_row})

print("Nombre de lignes après jointure leads+CRM:", len(merged))
merged[:2]

## 2) Réaliser une analyse univariée et bivariée

### 2.1 Analyse univariée — Variables quantitatives
Mesures : moyenne, médiane, minimum, maximum, dispersion.

In [ ]:
# KPI par canal (base quantitative)
kpi_rows = []
for camp in campaigns:
    row = dict(camp)
    row["ctr"] = row["clicks"] / row["impressions"]
    row["conversion_rate"] = row["conversions"] / row["clicks"]
    row["cpl"] = row["cost"] / row["conversions"]

    leads_count = sum(1 for x in merged if x["channel"] == row["channel"])
    clients_count = sum(1 for x in merged if x["channel"] == row["channel"] and x["status"] == "Client")
    row["leads"] = leads_count
    row["clients"] = clients_count
    row["cost_per_lead"] = row["cost"] / leads_count if leads_count else None
    row["cost_per_client"] = row["cost"] / clients_count if clients_count else None
    kpi_rows.append(row)

quant_vars = ["cost", "impressions", "clicks", "conversions", "ctr", "conversion_rate", "cpl"]
quant_summary = []
for var in quant_vars:
    values = [float(r[var]) for r in kpi_rows]
    quant_summary.append({
        "variable": var,
        "mean": mean(values),
        "median": median(values),
        "min": min(values),
        "max": max(values),
        "dispersion_range": max(values) - min(values),
    })

quant_summary

### 2.2 Analyse univariée — Variables qualitatives
Mesures : fréquences, répartition, proportions.

In [ ]:
def freq_table(values: list[str], label: str):
    total = len(values)
    counts = Counter(values)
    return [
        {label: k, "count": v, "proportion": round(v / total, 4)}
        for k, v in counts.items()
    ]

freq_channel = freq_table([r["channel"] for r in merged], "channel")
freq_status = freq_table([r["status"] for r in merged], "status")
freq_size = freq_table([r["company_size"] for r in merged], "company_size")
freq_sector = freq_table([r["sector"] for r in merged], "sector")
freq_region = freq_table([r["region"] for r in merged], "region")

freq_channel, freq_status

### 2.3 Analyse bivariée
Croisements pertinents métier :
- `channel × status`
- `company_size × status`
- `sector × status`
- `region × client_rate`

In [ ]:
status_by_channel = defaultdict(int)
status_by_size = defaultdict(int)
status_by_sector = defaultdict(int)

for row in merged:
    status_by_channel[(row["channel"], row["status"])] += 1
    status_by_size[(row["company_size"], row["status"])] += 1
    status_by_sector[(row["sector"], row["status"])] += 1

channels = sorted({r["channel"] for r in merged})
statuses = sorted({r["status"] for r in merged})
sizes = sorted({r["company_size"] for r in merged})
sectors = sorted({r["sector"] for r in merged})

rows_channel = []
for c in channels:
    row = {"channel": c}
    for s in statuses:
        row[s] = status_by_channel[(c, s)]
    rows_channel.append(row)

rows_size = []
for size in sizes:
    row = {"company_size": size}
    for s in statuses:
        row[s] = status_by_size[(size, s)]
    rows_size.append(row)

rows_sector = []
for sec in sectors:
    row = {"sector": sec}
    for s in statuses:
        row[s] = status_by_sector[(sec, s)]
    rows_sector.append(row)

region_totals = Counter(r["region"] for r in merged)
region_clients = Counter(r["region"] for r in merged if r["status"] == "Client")
region_rows = []
for region, total in sorted(region_totals.items()):
    region_rows.append({
        "region": region,
        "leads": total,
        "clients": region_clients[region],
        "client_rate": round(region_clients[region] / total, 4),
    })

rows_channel, rows_size, rows_sector, region_rows

### 2.4 Interprétations utiles pour la décision

In [ ]:
# Tri KPI par performance
best_ctr = max(kpi_rows, key=lambda x: x["ctr"])
best_conv = max(kpi_rows, key=lambda x: x["conversion_rate"])
best_cpl = min(kpi_rows, key=lambda x: x["cpl"])

insights = {
    "best_ctr_channel": best_ctr["channel"],
    "best_ctr": round(best_ctr["ctr"] * 100, 2),
    "best_conversion_channel": best_conv["channel"],
    "best_conversion": round(best_conv["conversion_rate"] * 100, 2),
    "best_cpl_channel": best_cpl["channel"],
    "best_cpl_eur": round(best_cpl["cpl"], 2),
}

insights

## 3) Produire des visualisations pertinentes (3 à 6)

> Chaque graphique répond à une question métier claire.
> Ici : 5 visualisations en syntaxe Mermaid (portable dans Markdown/Jupyter rendu compatible).

### Visualisation 1 — Quel canal a le meilleur CTR ?
```mermaid
xychart-beta
    title "CTR par canal"
    x-axis ["Emailing", "Google Ads", "LinkedIn Ads"]
    y-axis "CTR" 0 --> 0.035
    bar [0.03, 0.0267, 0.022]
```

### Visualisation 2 — Quel canal convertit le mieux ?
```mermaid
xychart-beta
    title "Taux de conversion par canal"
    x-axis ["Emailing", "Google Ads", "LinkedIn Ads"]
    y-axis "Taux" 0 --> 0.10
    bar [0.0833, 0.0813, 0.0864]
```

### Visualisation 3 — Quel canal minimise le CPL ?
```mermaid
xychart-beta
    title "CPL (€) par canal"
    x-axis ["Emailing", "Google Ads", "LinkedIn Ads"]
    y-axis "€" 0 --> 45
    bar [10, 16.15, 40]
```

### Visualisation 4 — Répartition des statuts CRM par canal
```mermaid
xychart-beta
    title "Nombre de clients par canal"
    x-axis ["Emailing", "Google Ads", "LinkedIn Ads"]
    y-axis "Nb clients" 0 --> 4
    bar [0, 0, 3]
```

### Visualisation 5 — Taux de clients par région
```mermaid
xychart-beta
    title "Taux clients par région"
    x-axis ["Bretagne", "Grand Est", "IDF", "Hauts-de-France", "Nlle-Aquitaine", "Occitanie", "PACA", "ARA"]
    y-axis "Taux" 0 --> 1
    bar [0, 0, 0.67, 0, 0, 0, 1, 0]
```

## 4) Concevoir un tableau de bord synthétique

### KPI (3 à 6 max)
1. CTR moyen (3 canaux)
2. Taux de conversion moyen
3. CPL moyen
4. Canal meilleur CTR
5. Canal meilleur taux de conversion
6. Canal CPL le plus faible

In [ ]:
kpi_dashboard = {
    "ctr_moyen_pct": round(mean([r["ctr"] for r in kpi_rows]) * 100, 2),
    "conversion_moyenne_pct": round(mean([r["conversion_rate"] for r in kpi_rows]) * 100, 2),
    "cpl_moyen_eur": round(mean([r["cpl"] for r in kpi_rows]), 2),
    "meilleur_ctr": f"{best_ctr['channel']} ({round(best_ctr['ctr']*100,2)}%)",
    "meilleure_conversion": f"{best_conv['channel']} ({round(best_conv['conversion_rate']*100,2)}%)",
    "cpl_le_plus_faible": f"{best_cpl['channel']} ({round(best_cpl['cpl'],2)}€)",
}

kpi_dashboard

## 5) Rédiger une note d’analyse métier (1 à 2 pages)

### 5.1 Contexte et objectifs
NovaRetail (SaaS B2B) souhaite évaluer ses campagnes Emailing / Google Ads / LinkedIn Ads sur octobre 2025.
Objectifs : CTR par canal, taux de conversion, coût par lead/client, segmentation taille/secteur/région.

### 5.2 Résultats clés
- Emailing : meilleur CTR et meilleur CPL.
- LinkedIn Ads : meilleur taux de conversion.
- Répartition CRM observée : MQL=4, SQL=3, Client=3.

### 5.3 Interprétation métier
- Emailing = moteur d’acquisition rentable.
- LinkedIn Ads = canal de qualification plus premium mais coûteux.
- Google Ads = potentiel d’optimisation (ciblage, mots-clés, landing page).

### 5.4 Recommandations opérationnelles
1. Réallouer budget selon objectif (volume vs qualité).
2. Optimiser Google Ads par A/B tests et requêtes intention forte.
3. Prioriser les segments les plus transformants (taille, secteur, région).

## 6) Documenter un carnet technique

### Problèmes rencontrés / Solutions / Justification
1. **Formats hétérogènes** (`csv`, `json`, `xlsx` dans énoncé) → normalisation + pipeline unique.
2. **Périmètre temporel strict** → filtre date explicite.
3. **Jointure marketing/CRM** → clé `lead_id` et agrégation par `channel`.
4. **Homogénéité KPI** → calculs centralisés dans notebook/script.
5. **Lisibilité livrable** → sections numérotées + exports CSV réutilisables.

## 7) Exports finaux (tables d’analyse)

In [ ]:
write_csv(
    OUTPUT_DIR / "kpi_par_canal.csv",
    kpi_rows,
    [
        "campaign_id", "channel", "cost", "impressions", "clicks", "conversions",
        "ctr", "conversion_rate", "cpl", "leads", "clients", "cost_per_lead", "cost_per_client"
    ],
)
write_csv(OUTPUT_DIR / "analyse_univariee_quant.csv", quant_summary, list(quant_summary[0].keys()))
write_csv(OUTPUT_DIR / "frequences_channel.csv", freq_channel, ["channel", "count", "proportion"])
write_csv(OUTPUT_DIR / "frequences_status.csv", freq_status, ["status", "count", "proportion"])
write_csv(OUTPUT_DIR / "croisement_status_channel.csv", rows_channel, ["channel", *statuses])
write_csv(OUTPUT_DIR / "croisement_status_taille.csv", rows_size, ["company_size", *statuses])
write_csv(OUTPUT_DIR / "croisement_status_secteur.csv", rows_sector, ["sector", *statuses])
write_csv(OUTPUT_DIR / "taux_client_region.csv", region_rows, ["region", "leads", "clients", "client_rate"])

print("Exports générés dans:", OUTPUT_DIR)